# TD05 — Support Vector Machines

## Objectif

Sur un même jeu de données 2D, comparer le **perceptron** et le **SVM** (cours « 5 - Support Vector Machines ») :

- sur des données **séparables**, le perceptron s'arrête sur *une* droite séparatrice quelconque, alors que le SVM cherche *la* droite de **marge maximale** ;
- on code le SVM à marge souple en minimisant par **descente de gradient** la perte

$$E(w, b) = \frac{\lambda}{2}\|w\|^2 + \frac{1}{N} \sum_{i=1}^N \max\big(0,\ 1 - t_i z_i\big), \qquad z_i = w \cdot x_i + b.$$

**Notations**
- Cible $t$, sortie (prédiction) $y$, score $z = w \cdot x + b$ (biais en $+b$, comme au TD02).
- ⚠️ **Cibles $t \in \{-1, +1\}$** dans tout ce TD, et non plus $\{0, 1\}$ comme au TD01. Un point est bien classé si et seulement si $t\,z > 0$.
- Dans le code : `lr` pour le pas d'apprentissage, `lam` pour $\lambda$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Alias de types pour les tableaux numpy
Vec = np.ndarray[tuple[int], np.dtype[np.float64]]        # vecteur 1D de flottants
Mat = np.ndarray[tuple[int, int], np.dtype[np.float64]]   # matrice 2D de flottants
VecInt = np.ndarray[tuple[int], np.dtype[np.int64]]       # vecteur 1D d'entiers (cibles ±1)

## Préambule : données et outils fournis

Rien à coder dans cette section : exécutez les cellules et lisez le code.

### Génération des données

`gen_donnees(n, ecart, seed, n_bruit)` tire `n` points 2D en deux classes ($t = +1$ et $t = -1$), de part et d'autre de la droite $x_1 + x_2 = 0$, en laissant une **bande vide** de largeur `ecart` entre les deux classes : le jeu est donc **séparable**, mais de justesse. Les classes sont étirées le long de la frontière.

`n_bruit` inverse l'étiquette de `n_bruit` points tirés au hasard (utilisé plus tard). Avec la même graine, les points `X` sont les mêmes, seules les étiquettes changent.

On utilise **peu** de points d'entraînement (24) et **beaucoup** de points de test (500), tirés de la même distribution : on pourra ainsi mesurer finement la généralisation.

In [ ]:
def gen_donnees(n: int, ecart: float, seed: int, n_bruit: int = 0) -> tuple[Mat, VecInt]:
    rng = np.random.default_rng(seed)
    u = np.array([1.0, 1.0]) / np.sqrt(2)    # normale à la vraie frontière x1 + x2 = 0
    v = np.array([-1.0, 1.0]) / np.sqrt(2)   # direction de la vraie frontière
    t = np.where(np.arange(n) < n // 2, 1, -1)
    a = t * (ecart / 2 + np.abs(rng.normal(0.0, 1.0, n)))   # distance signée à la frontière
    c = rng.normal(0.0, 2.0, n)                              # position le long de la frontière
    X = a[:, None] * u + c[:, None] * v
    perm = rng.permutation(n)
    X, t = X[perm], t[perm]
    if n_bruit > 0:
        idx = rng.choice(n, size=n_bruit, replace=False)
        t = t.copy()
        t[idx] = -t[idx]
    return X, t


ECART = 0.6
X_train, t_train = gen_donnees(24, ECART, seed=4)
X_test, t_test = gen_donnees(500, ECART, seed=1000)
print("train :", X_train.shape, "test :", X_test.shape, "cibles :", np.unique(t_train))

### Affichage

`plot_modele(ax, X, t, w, b, ...)` affiche les points (bleu : $t = +1$, orange : $t = -1$) et la frontière $z = w \cdot x + b = 0$ (trait plein). Options :
- `marges=True` : les droites $z = \pm 1$ en pointillés ;
- `supports=True` : entoure les points tels que $t\,z \le 1 + \varepsilon$ (vecteurs de support), avec $\varepsilon$ = `eps`.

In [ ]:
def plot_modele(ax, X, t, w, b, titre="", marges=False, supports=False, eps=0.05):
    taille = 25 if len(X) <= 100 else 8
    ax.scatter(X[t == 1, 0], X[t == 1, 1], c="tab:blue", edgecolors="k", s=taille, label="t = +1")
    ax.scatter(X[t == -1, 0], X[t == -1, 1], c="tab:orange", edgecolors="k", s=taille, label="t = -1")
    lim = np.abs(X).max() + 0.5
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    w = np.asarray(w, dtype=float)
    if np.linalg.norm(w) > 0:
        # point de la droite w·x + b = niveau le plus proche de l'origine
        def droite(niveau, **style):
            p0 = (niveau - b) * w / (w @ w)
            ax.axline(p0, p0 + np.array([-w[1], w[0]]), **style)
        droite(0.0, color="k", lw=2)
        if marges:
            droite(1.0, color="k", lw=1, ls="--")
            droite(-1.0, color="k", lw=1, ls="--")
    if supports:
        sv = t * (X @ w + b) <= 1 + eps
        ax.scatter(X[sv, 0], X[sv, 1], s=180, facecolors="none", edgecolors="crimson", lw=1.5,
                   label="vecteurs de support")
    if titre:
        ax.set_title(titre, fontsize=10)


fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_modele(axes[0], X_train, t_train, np.array([1.0, 1.0]), 0.0, titre="train (24 pts) et vraie frontière")
plot_modele(axes[1], X_test, t_test, np.array([1.0, 1.0]), 0.0, titre="test (500 pts) et vraie frontière")
axes[0].legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

### Perceptron et accuracy

Le perceptron a été codé au TD01 : on en fournit ici une version vectorielle, avec cibles $\pm 1$ et biais en $+b$. Pour chaque point mal classé ($t\,z \le 0$) : $w \leftarrow w + \texttt{lr}\; t\,x$, $b \leftarrow b + \texttt{lr}\; t$ (règle de Rosenblatt, cours §4).

- `fit(X, t, lr, max_epochs, seed)` parcourt les points dans un ordre mélangé selon `seed` (nouvel ordre à chaque époque), s'arrête dès qu'une époque se fait sans erreur et renvoie le nombre d'époques effectuées. `self.erreurs` contient le nombre d'erreurs de chaque époque.
- `accuracy(y, t)` renvoie la proportion (entre 0 et 1) de prédictions correctes.

In [ ]:
class Perceptron:
    def __init__(self, d: int):
        self.w = np.zeros(d)
        self.b = 0.0
        self.erreurs = []

    def predict(self, X: Mat) -> VecInt:
        return np.where(X @ self.w + self.b >= 0, 1, -1)

    def fit(self, X: Mat, t: VecInt, lr: float = 0.1, max_epochs: int = 100, seed: int = 0) -> int:
        rng = np.random.default_rng(seed)
        self.erreurs = []
        for epoque in range(1, max_epochs + 1):
            n_err = 0
            for i in rng.permutation(len(X)):
                if t[i] * (X[i] @ self.w + self.b) <= 0:
                    self.w = self.w + lr * t[i] * X[i]
                    self.b = self.b + lr * t[i]
                    n_err += 1
            self.erreurs.append(n_err)
            if n_err == 0:
                return epoque
        return max_epochs


def accuracy(y: VecInt, t: VecInt) -> float:
    return float(np.mean(y == t))

## 1. Données séparables : le perceptron et la marge

### 1.1 Marge géométrique

La marge d'un séparateur $(w, b)$ sur un jeu $(X, t)$ est la distance (signée) du point le plus proche à la frontière (cours §2.1) :

$$\text{marge}(w, b) = \min_i \frac{t_i\,(w \cdot x_i + b)}{\|w\|}$$

Elle est négative si au moins un point est mal classé.

> **Exercice 1** — Complétez `marge(w, b, X, t)`, en une ou deux lignes vectorisées (sans boucle).

In [ ]:
def marge(w: Vec, b: float, X: Mat, t: VecInt) -> float:
    # BEGIN
    # raise NotImplementedError
    ## Indice : scores z = X @ w + b ; np.min et np.linalg.norm
    z = X @ w + b
    return float(np.min(t * z) / np.linalg.norm(w))
    # END

#### Vérification de `marge`

Trois points sur l'axe $x_1$, frontière $x_1 = 0$ : le point le plus proche est à distance 1. Multiplier $(w, b)$ par 2 ne change pas la marge ; décaler la frontière en $x_1 = 2$ classe mal le point $(1, 0)$.

In [ ]:
def check_marge():
    X = np.array([[1.0, 0.0], [3.0, 0.0], [-2.0, 0.0]])
    t = np.array([1, 1, -1])
    assert np.isclose(marge(np.array([1.0, 0.0]), 0.0, X, t), 1.0)
    assert np.isclose(marge(np.array([2.0, 0.0]), 0.0, X, t), 1.0)   # invariance d'échelle
    assert marge(np.array([1.0, 0.0]), -2.0, X, t) < 0                # point mal classé
    print("marge check — OK")


check_marge()

### 1.2 Le perceptron, plusieurs fois

> **Exercice 2** — Pour chaque graine `g` de `graines` : créez un `Perceptron(2)`, entraînez-le sur `(X_train, t_train)` avec `seed=g`, ajoutez-le à la liste `perceptrons`, et affichez le nombre d'époques, sa marge sur le jeu d'entraînement et son accuracy sur le jeu de test.

In [ ]:
graines = [0, 1, 2, 3, 4]
perceptrons = []
# BEGIN
# raise NotImplementedError
## Indice : n_ep = p.fit(X_train, t_train, seed=g) ; puis marge(p.w, p.b, X_train, t_train)
for g in graines:
    p = Perceptron(2)
    n_ep = p.fit(X_train, t_train, seed=g)
    perceptrons.append(p)
    print(f"graine {g} : {n_ep} époques, marge = {marge(p.w, p.b, X_train, t_train):.3f}, "
          f"accuracy test = {accuracy(p.predict(X_test), t_test):.3f}")
# END

#### Vérification

In [ ]:
def check_perceptrons():
    assert len(perceptrons) == len(graines), "il faut un perceptron par graine"
    for p in perceptrons:
        assert accuracy(p.predict(X_train), t_train) == 1.0, "chaque perceptron doit séparer le train"
    assert len({tuple(p.w) for p in perceptrons}) > 1, "utilisez seed=g dans fit"
    print("perceptrons check — OK")


check_perceptrons()

La cellule suivante affiche chaque perceptron : en haut sur le jeu d'entraînement, en bas sur le jeu de test.

In [ ]:
fig, axes = plt.subplots(2, len(perceptrons), figsize=(4 * len(perceptrons), 8))
for g, p, ax_tr, ax_te in zip(graines, perceptrons, axes[0], axes[1]):
    plot_modele(ax_tr, X_train, t_train, p.w, p.b,
                titre=f"graine {g} — marge {marge(p.w, p.b, X_train, t_train):.3f}")
    plot_modele(ax_te, X_test, t_test, p.w, p.b,
                titre=f"test — accuracy {accuracy(p.predict(X_test), t_test):.3f}")
plt.tight_layout()
plt.show()

> **Question** — Les cinq perceptrons ont-ils trouvé la même droite ? Ont-ils tous convergé ? Que vaut leur marge, comparée à la largeur `ECART` = 0.6 de la bande vide ? Comment se comportent-ils sur le jeu de test ?

*Votre réponse :*

# TOCLEAR

- Les cinq perceptrons convergent tous (en 2 ou 3 époques) : ils séparent parfaitement les 24 points d'entraînement. Mais ils trouvent **cinq droites différentes** : le résultat dépend de l'ordre de présentation des points (la graine).
- Leurs marges sont petites (entre 0.07 et 0.18 environ) : chaque droite **frôle** au moins un point d'entraînement. Une droite qui passerait au milieu de la bande vide aurait une marge d'au moins 0.3 (la moitié de l'écart), et sans doute davantage avec ces 24 points.
- Sur le test, aucune n'est parfaite (accuracy entre 94 % et 98 %) : les points de test proches de la frontière, du côté où la droite frôle les données, sont mal classés. Le perceptron s'arrête dès que l'erreur d'entraînement est nulle ; rien ne le pousse à choisir une droite « bien placée ».

## 2. SVM par descente de gradient

### 2.1 La classe `SVM`

On minimise la perte du cours (§3.2), où le biais $b$ n'est **pas** régularisé :

$$E(w, b) = \frac{\lambda}{2}\|w\|^2 + \frac{1}{N} \sum_{i=1}^N \max\big(0,\ 1 - t_i z_i\big), \qquad z_i = w \cdot x_i + b$$

Son sous-gradient (§3.3) ne fait intervenir que les points qui **violent leur marge** ($t_i z_i < 1$, inégalité **stricte**) :

$$\frac{\partial E}{\partial w} = \lambda w - \frac{1}{N} \sum_{i \,:\, t_i z_i < 1} t_i\, x_i \qquad\qquad \frac{\partial E}{\partial b} = -\frac{1}{N} \sum_{i \,:\, t_i z_i < 1} t_i$$

Toutes les méthodes sont **vectorisées** (pas de boucle sur les points). On part de $w = 0$, $b = 0$.

> **Exercice 3a** — Complétez `scores(X)` (renvoie le vecteur des $z_i$) et `predict(X)` (renvoie $+1$ si $z \ge 0$, $-1$ sinon).

> **Exercice 3b** — Complétez `loss(X, t)` : la valeur de $E(w, b)$.

> **Exercice 3c** — Complétez `gradient(X, t)` : renvoie le couple `(dw, db)` $= \left(\frac{\partial E}{\partial w}, \frac{\partial E}{\partial b}\right)$.

> **Exercice 3d** — Complétez `fit(X, t, lr, epochs)` : descente de gradient **batch** ($w \leftarrow w - \texttt{lr}\; dw$, $b \leftarrow b - \texttt{lr}\; db$), répétée `epochs` fois ; à la fin de chaque époque, ajoutez la perte $E$ à `self.pertes`. Renvoie `self`.

In [ ]:
class SVM:
    def __init__(self, d: int, lam: float = 0.01):
        self.w = np.zeros(d)
        self.b = 0.0
        self.lam = lam
        self.pertes = []

    def scores(self, X: Mat) -> Vec:
        # BEGIN
        # raise NotImplementedError
        ## Indice : un seul produit matrice-vecteur (@), plus le biais
        return X @ self.w + self.b
        # END

    def predict(self, X: Mat) -> VecInt:
        # BEGIN
        # raise NotImplementedError
        ## Indice : np.where(condition, 1, -1)
        return np.where(self.scores(X) >= 0, 1, -1)
        # END

    def loss(self, X: Mat, t: VecInt) -> float:
        # BEGIN
        # raise NotImplementedError
        ## Indice : hinge = np.maximum(0, 1 - t * z) ; régularisation lam / 2 * w·w
        hinge = np.maximum(0.0, 1 - t * self.scores(X))
        return float(self.lam / 2 * (self.w @ self.w) + hinge.mean())
        # END

    def gradient(self, X: Mat, t: VecInt) -> tuple[Vec, float]:
        # BEGIN
        # raise NotImplementedError
        ## Indice : masque viol = t * z < 1 ; t[viol] @ X[viol] calcule la somme des t_i x_i
        N = len(X)
        viol = t * self.scores(X) < 1
        dw = self.lam * self.w - (t[viol] @ X[viol]) / N
        db = -float(t[viol].sum()) / N
        return dw, db
        # END

    def fit(self, X: Mat, t: VecInt, lr: float = 0.1, epochs: int = 1000) -> "SVM":
        self.pertes = []
        # BEGIN
        # raise NotImplementedError
        ## Indice : dw, db = self.gradient(X, t) ; mise à jour de w et b ; self.pertes.append(self.loss(X, t))
        for _ in range(epochs):
            dw, db = self.gradient(X, t)
            self.w = self.w - lr * dw
            self.b = self.b - lr * db
            self.pertes.append(self.loss(X, t))
        return self
        # END

#### Vérification de `scores` et `predict`

In [ ]:
def check_scores_predict():
    s = SVM(2)
    s.w = np.array([1.0, -2.0])
    s.b = 1.0
    X = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [-1.0, 0.0]])
    assert np.array_equal(s.scores(X), np.array([1.0, 0.0, -1.0, 0.0]))
    assert np.array_equal(s.predict(X), np.array([1, 1, -1, 1]))   # z = 0 -> +1
    print("scores / predict check — OK")


check_scores_predict()

#### Vérification de `loss`

Calcul à la main : avec $w = (1, 0)$, $b = 0$ et trois points de cible $+1$, les $t_i z_i$ valent $2,\ 0.5,\ -1$, donc les pertes hinge $0,\ 0.5,\ 2$ (moyenne $2.5/3$) ; la régularisation vaut $\frac{0.1}{2} \times 1 = 0.05$. Avec $b = 1$ (même $w$), les $t_i z_i$ valent $3,\ 1.5,\ 0$ : pertes hinge $0,\ 0,\ 1$, et la régularisation ne change pas ($b$ n'est pas régularisé).

In [ ]:
def check_loss():
    s = SVM(2, lam=0.1)
    s.w = np.array([1.0, 0.0])
    s.b = 0.0
    X = np.array([[2.0, 0.0], [0.5, 0.0], [-1.0, 0.0]])
    t = np.array([1, 1, 1])
    assert np.isclose(s.loss(X, t), 2.5 / 3 + 0.05), s.loss(X, t)
    s.b = 1.0
    assert np.isclose(s.loss(X, t), 1 / 3 + 0.05), "b ne doit pas être régularisé"
    print("loss check — OK")


check_loss()

#### Vérification de `gradient`

On compare `gradient` à des **différences finies** : $\frac{\partial E}{\partial w_j} \approx \frac{E(w + \epsilon e_j) - E(w - \epsilon e_j)}{2\epsilon}$, sur des données et des paramètres aléatoires (aucun point n'est exactement sur sa marge, là où la hinge n'est pas dérivable).

In [ ]:
def check_gradient(eps=1e-6):
    rng = np.random.default_rng(0)
    X = rng.normal(size=(20, 2))
    t = np.where(rng.random(20) < 0.5, 1, -1)
    s = SVM(2, lam=0.3)
    s.w = rng.normal(size=2)
    s.b = float(rng.normal())
    assert np.min(np.abs(t * s.scores(X) - 1)) > 1e-3   # pas de point sur t z = 1
    dw, db = s.gradient(X, t)
    w0, b0 = s.w.copy(), s.b
    for j in range(2):
        e = np.zeros(2)
        e[j] = eps
        s.w = w0 + e
        Ep = s.loss(X, t)
        s.w = w0 - e
        Em = s.loss(X, t)
        s.w = w0
        assert np.isclose(dw[j], (Ep - Em) / (2 * eps), atol=1e-6), f"dw[{j}] incorrect"
    s.b = b0 + eps
    Ep = s.loss(X, t)
    s.b = b0 - eps
    Em = s.loss(X, t)
    s.b = b0
    assert np.isclose(db, (Ep - Em) / (2 * eps), atol=1e-6), "db incorrect"
    print("gradient check — OK")


check_gradient()

#### Vérification de `fit`

Deux époques simulées à la main, avec $\lambda = 0.5$, `lr` $= 0.5$, $X = \{(2, 0), (0, 2)\}$, $t = (+1, -1)$ :
- époque 1 : $w = 0$, donc $t_i z_i = 0 < 1$ pour les deux points : $dw = -\frac{1}{2}\big((2, 0) - (0, 2)\big) = (-1, 1)$, $db = -\frac{1}{2}(1 - 1) = 0$, d'où $w = (0.5, -0.5)$, $b = 0$. Ensuite $t_i z_i = 1$ pour les deux points : perte hinge nulle, $E = \frac{0.5}{2} \times 0.5 = 0.125$ ;
- époque 2 : $t_i z_i = 1$ n'est **pas** $< 1$ : seul le terme $\lambda w$ reste, $dw = (0.25, -0.25)$, $db = 0$, d'où $w = (0.375, -0.375)$, $b = 0$. Alors $t_i z_i = 0.75$ : hinge $0.25$ pour chaque point, et $E = 0.25 + \frac{0.5}{2} \times 0.28125 = 0.3203125$.

On vérifie ensuite que `fit` sépare le jeu d'entraînement et fait baisser la perte.

In [ ]:
def check_fit():
    s = SVM(2, lam=0.5)
    X = np.array([[2.0, 0.0], [0.0, 2.0]])
    t = np.array([1, -1])
    s.fit(X, t, lr=0.5, epochs=2)
    assert np.array_equal(s.w, np.array([0.375, -0.375])), s.w
    assert s.b == 0.0, s.b
    assert s.pertes == [0.125, 0.3203125], s.pertes

    s = SVM(2).fit(X_train, t_train)
    assert len(s.pertes) == 1000
    assert s.pertes[-1] < s.pertes[0]
    assert accuracy(s.predict(X_train), t_train) == 1.0
    print("fit check — OK")


check_fit()

La cellule suivante entraîne un SVM sur `(X_train, t_train)` et affiche sa courbe de perte : elle doit décroître puis se stabiliser.

In [ ]:
s = SVM(2, lam=0.01).fit(X_train, t_train, lr=0.1, epochs=1000)
print(f"perte initiale : {s.pertes[0]:.4f}, perte finale : {s.pertes[-1]:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(s.pertes)
ax.set_xlabel("époque")
ax.set_ylabel("E(w, b)")
ax.set_title("Perte du SVM")
ax.grid(True, alpha=0.3)
plt.show()